In [2]:
import sys
print(sys.executable)

C:\Users\N01SC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


In [1]:
from pypdf import PdfReader

In [ ]:
import sys
print(sys.executable)

In [2]:
pdf_path = "../data/sample.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print(text[:1000])

 
 
Eastern Mediterranean 
University 
School of Computing and Technology 
Claud Computing Applications  
 
Prepared by   
Abdulrahman Ahmed 22902545 
Osman Eid 23704362 
Mohamed Allafi 22901510 
Abdulmalik Khalid 22901037 
Ali Mohammed 23704831 
 
 
 
 

Section One: Project Objective 
The objective of this project is to design, build, and test a simple RESTful Web 
Service using ASP.NET Core Web API. The application chosen for this 
assignment is a Clinic Appointment Management API, named ClinicApi, which 
allows a client to manage patient appointments through a set of HTTP endpoints.  
The main goal of the project is to understand how a Web API is structured 
internally — from the data model, to the business logic layer, to the controller that 
exposes the endpoints — and to learn how to test and verify that a REST API 
behaves correctly using an API testing tool such as Postman. The project 
demonstrates the full lifecycle of a resource (an appointment) through Create, 
Read, Updat

In [3]:
len(reader.pages)

19

In [4]:
len(text)

16403

In [5]:
def split_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

    return chunks

In [6]:
chunks = split_text(text)

print("Number of chunks:", len(chunks))
print(chunks[0])

Number of chunks: 26
 
 
Eastern Mediterranean 
University 
School of Computing and Technology 
Claud Computing Applications  
 
Prepared by   
Abdulrahman Ahmed 22902545 
Osman Eid 23704362 
Mohamed Allafi 22901510 
Abdulmalik Khalid 22901037 
Ali Mohammed 23704831 
 
 
 
 

Section One: Project Objective 
The objective of this project is to design, build, and test a simple RESTful Web 
Service using ASP.NET Core Web API. The application chosen for this 
assignment is a Clinic Appointment Management API, named ClinicApi, which 
allows a client to manage patient appointments through a set of HTTP endpoints.  
The main goal of the project is to understand how a Web API is structured 
internally — from the data model, to the business logic layer, to the controller that 
exposes the endpoints — and to learn how t


In [7]:
print(chunks[-1])

re) so that data persists between application restarts, and adding 
input validation to reject malformed requests before they reach the service layer.  



## Text Chunking

The extracted PDF text was split into smaller chunks before creating embeddings.

Chunking is important because RAG systems retrieve the most relevant parts of a document instead of sending the entire document to the model.

An overlap was used between chunks to avoid losing context when important information appears near the boundary between two chunks.

In [8]:
from sentence_transformers import SentenceTransformer

c:\Users\N01SC\Desktop\AI-Projects\pdf-question-answering-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\N01SC\Desktop\AI-Projects\pdf-question-answering-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\N01SC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2379.3

In [10]:
embeddings = embedding_model.encode(chunks)

embeddings.shape

(26, 384)

In [11]:
embeddings[0][:10]

array([-0.07109445,  0.05852255, -0.08572187, -0.06261418, -0.10484356,
       -0.04720397,  0.01036855,  0.05266438,  0.00169758, -0.03909507],
      dtype=float32)

## Embeddings

Each text chunk was converted into an embedding using the all-MiniLM-L6-v2 SentenceTransformer model.

Embeddings are numerical representations of text meaning. They allow the system to compare the user's question with document chunks and retrieve the most relevant information.

The embedding matrix shape shows the number of chunks and the number of dimensions in each embedding vector.

In [12]:
embeddings.shape

(26, 384)

In [13]:
import faiss
import numpy as np

In [14]:
embeddings_np = np.array(embeddings).astype("float32")

In [15]:
dimension = embeddings_np.shape[1]

index = faiss.IndexFlatL2(dimension)

In [16]:
index.add(embeddings_np)

In [17]:
index.ntotal

26

In [18]:
question = "What is the objective of the project?"

question_embedding = embedding_model.encode([question])
question_embedding_np = np.array(question_embedding).astype("float32")

distances, indices = index.search(question_embedding_np, k=3)

print(indices)
print(distances)

[[0 2 3]]
[[1.6719604 1.721622  1.7289686]]


In [19]:
for i, idx in enumerate(indices[0]):
    print(f"Result {i+1}")
    print("Distance:", distances[0][i])
    print(chunks[idx])
    print("-" * 80)

Result 1
Distance: 1.6719604
 
 
Eastern Mediterranean 
University 
School of Computing and Technology 
Claud Computing Applications  
 
Prepared by   
Abdulrahman Ahmed 22902545 
Osman Eid 23704362 
Mohamed Allafi 22901510 
Abdulmalik Khalid 22901037 
Ali Mohammed 23704831 
 
 
 
 

Section One: Project Objective 
The objective of this project is to design, build, and test a simple RESTful Web 
Service using ASP.NET Core Web API. The application chosen for this 
assignment is a Clinic Appointment Management API, named ClinicApi, which 
allows a client to manage patient appointments through a set of HTTP endpoints.  
The main goal of the project is to understand how a Web API is structured 
internally — from the data model, to the business logic layer, to the controller that 
exposes the endpoints — and to learn how t
--------------------------------------------------------------------------------
Result 2
Distance: 1.721622
oint returns the expected status code and 
payload. 
Section 

## Vector Search with FAISS

The chunk embeddings were stored in a FAISS index to enable semantic search.

When a user asks a question, the question is converted into an embedding using the same embedding model. FAISS then compares the question embedding with the stored chunk embeddings and returns the most relevant chunks.

This retrieval step is the core of the RAG system.

In [20]:
def retrieve_chunks(question, top_k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding_np = np.array(question_embedding).astype("float32")

    distances, indices = index.search(question_embedding_np, k=top_k)

    retrieved = []

    for i, idx in enumerate(indices[0]):
        retrieved.append({
            "chunk": chunks[idx],
            "distance": distances[0][i]
        })

    return retrieved

In [21]:
question = "What is the objective of the project?"

retrieved_chunks = retrieve_chunks(question)

for item in retrieved_chunks:
    print("Distance:", item["distance"])
    print(item["chunk"][:700])
    print("-" * 80)

Distance: 1.6719604
 
 
Eastern Mediterranean 
University 
School of Computing and Technology 
Claud Computing Applications  
 
Prepared by   
Abdulrahman Ahmed 22902545 
Osman Eid 23704362 
Mohamed Allafi 22901510 
Abdulmalik Khalid 22901037 
Ali Mohammed 23704831 
 
 
 
 

Section One: Project Objective 
The objective of this project is to design, build, and test a simple RESTful Web 
Service using ASP.NET Core Web API. The application chosen for this 
assignment is a Clinic Appointment Management API, named ClinicApi, which 
allows a client to manage patient appointments through a set of HTTP endpoints.  
The main goal of the project is to understand how a Web API is structured 
internally — from the data mo
--------------------------------------------------------------------------------
Distance: 1.721622
oint returns the expected status code and 
payload. 
Section Two: Technologies Used 
The following technologies and tools were used to build, run, and test the Web 
Service: 
● AS

In [22]:
def answer_question(question, top_k=3):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)

    answer = retrieved_chunks[0]["chunk"]

    return answer, retrieved_chunks

In [23]:
question = "What is the objective of the project?"

answer, sources = answer_question(question)

print("Question:", question)
print("\nAnswer based on document:")
print(answer[:1000])

Question: What is the objective of the project?

Answer based on document:
 
 
Eastern Mediterranean 
University 
School of Computing and Technology 
Claud Computing Applications  
 
Prepared by   
Abdulrahman Ahmed 22902545 
Osman Eid 23704362 
Mohamed Allafi 22901510 
Abdulmalik Khalid 22901037 
Ali Mohammed 23704831 
 
 
 
 

Section One: Project Objective 
The objective of this project is to design, build, and test a simple RESTful Web 
Service using ASP.NET Core Web API. The application chosen for this 
assignment is a Clinic Appointment Management API, named ClinicApi, which 
allows a client to manage patient appointments through a set of HTTP endpoints.  
The main goal of the project is to understand how a Web API is structured 
internally — from the data model, to the business logic layer, to the controller that 
exposes the endpoints — and to learn how t


## Local Question Answering

A simple local question-answering function was created using retrieval only.

The system retrieves the most relevant chunks from the PDF and returns the best matching chunk as the answer. This provides a basic local RAG-style workflow without requiring an external LLM API.

In a later upgrade, an LLM can be added to generate a more natural answer from the retrieved chunks.

In [24]:
question = "What application was built in the project?"

answer, sources = answer_question(question)

print("Question:", question)
print("\nAnswer based on document:")
print(answer[:1000])

Question: What application was built in the project?

Answer based on document:
oint returns the expected status code and 
payload. 
Section Two: Technologies Used 
The following technologies and tools were used to build, run, and test the Web 
Service: 
● ASP.NET Core Web API (C#) – used to build the RESTful Web Service. 
● Visual Studio / Visual Studio Code – development environment used to 
write and debug the project. 
● .NET SDK – used to compile and run the API on a local Kestrel web server 
(http://localhost:5109). 
● JSON (JavaScript Object Notation) – the data format used for all request and 
response bodies exchanged with the API. 
● Postman – used as the API Testing tool to send HTTP requests (GET, 
POST, PUT, DELETE) to the API's endpoints and inspect the responses.  
● Dependency Injection (built into ASP.NET Core) – used to register and 
inject the Appoi


In [25]:
def split_text_by_paragraphs(text, chunk_size=1000, overlap=200):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    
    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        if len(current_chunk) + len(paragraph) <= chunk_size:
            current_chunk += paragraph + " "
        else:
            chunks.append(current_chunk.strip())

            overlap_text = current_chunk[-overlap:]
            current_chunk = overlap_text + " " + paragraph + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [26]:
chunks = split_text_by_paragraphs(text)

print("Number of chunks:", len(chunks))
print(chunks[0])

Number of chunks: 20
Eastern Mediterranean University School of Computing and Technology Claud Computing Applications Prepared by Abdulrahman Ahmed 22902545 Osman Eid 23704362 Mohamed Allafi 22901510 Abdulmalik Khalid 22901037 Ali Mohammed 23704831 Section One: Project Objective The objective of this project is to design, build, and test a simple RESTful Web Service using ASP.NET Core Web API. The application chosen for this assignment is a Clinic Appointment Management API, named ClinicApi, which allows a client to manage patient appointments through a set of HTTP endpoints. The main goal of the project is to understand how a Web API is structured internally — from the data model, to the business logic layer, to the controller that exposes the endpoints — and to learn how to test and verify that a REST API behaves correctly using an API testing tool such as Postman. The project demonstrates the full lifecycle of a resource (an appointment) through Create,


In [27]:
embeddings = embedding_model.encode(chunks)
embeddings_np = np.array(embeddings).astype("float32")

dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

print("Chunks stored in FAISS:", index.ntotal)

Chunks stored in FAISS: 20


In [28]:
question = "What application was built in the project?"

answer, sources = answer_question(question)

print("Question:", question)
print("\nAnswer based on document:")
print(answer[:1000])

Question: What application was built in the project?

Answer based on document:
● Visual Studio / Visual Studio Code – development environment used to write and debug the project. ● .NET SDK – used to compile and run the API on a local Kestrel web server (http://localhost:5109).  ● JSON (JavaScript Object Notation) – the data format used for all request and response bodies exchanged with the API. ● Postman – used as the API Testing tool to send HTTP requests (GET, POST, PUT, DELETE) to the API's endpoints and inspect the responses. ● Dependency Injection (built into ASP.NET Core) – used to register and inject the AppointmentService into the controller through the IAppointmentService interface. Although the API in this project was implemented as a REST API rather than a SOAP API, both approaches were researched as part of the assignment. REST relies on standard HTTP Methods and lightweight JSON payloads, while a SOAP API uses XML-based messages and a formal Service Contract described by

In [29]:
questions = [
    "What is the objective of the project?",
    "What is the name of the application?",
    "What is ClinicApi?",
    "What does the Clinic Appointment Management API do?"
]

for q in questions:
    answer, sources = answer_question(q)
    print("Question:", q)
    print("Answer:")
    print(answer[:700])
    print("-" * 80)

Question: What is the objective of the project?
Answer:
Eastern Mediterranean University School of Computing and Technology Claud Computing Applications Prepared by Abdulrahman Ahmed 22902545 Osman Eid 23704362 Mohamed Allafi 22901510 Abdulmalik Khalid 22901037 Ali Mohammed 23704831 Section One: Project Objective The objective of this project is to design, build, and test a simple RESTful Web Service using ASP.NET Core Web API. The application chosen for this assignment is a Clinic Appointment Management API, named ClinicApi, which allows a client to manage patient appointments through a set of HTTP endpoints. The main goal of the project is to understand how a Web API is structured internally — from the data model, to the business logic layer, t
--------------------------------------------------------------------------------
Question: What is the name of the application?
Answer:
P Methods and lightweight JSON payloads, while a SOAP API uses XML-based messages and a formal Service Cont

In [30]:
question = "What application was built in the project?"

retrieved_chunks = retrieve_chunks(question, top_k=5)

for i, item in enumerate(retrieved_chunks):
    print(f"Source {i+1}")
    print("Distance:", item["distance"])
    print(item["chunk"][:700])
    print("-" * 80)

Source 1
Distance: 1.429074
● Visual Studio / Visual Studio Code – development environment used to write and debug the project. ● .NET SDK – used to compile and run the API on a local Kestrel web server (http://localhost:5109).  ● JSON (JavaScript Object Notation) – the data format used for all request and response bodies exchanged with the API. ● Postman – used as the API Testing tool to send HTTP requests (GET, POST, PUT, DELETE) to the API's endpoints and inspect the responses. ● Dependency Injection (built into ASP.NET Core) – used to register and inject the AppointmentService into the controller through the IAppointmentService interface. Although the API in this project was implemented as a REST API rather than a 
--------------------------------------------------------------------------------
Source 2
Distance: 1.5618713
to learn how to test and verify that a REST API behaves correctly using an API testing tool such as Postman. The project demonstrates the full lifecycle of a res

In [31]:
def answer_question(question, top_k=3):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)

    context = "\n\n".join([item["chunk"] for item in retrieved_chunks])

    return context, retrieved_chunks

In [32]:
def ask_question(question, top_k=5):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)

    print("Question:", question)
    print("\nMost relevant sources:\n")

    for i, item in enumerate(retrieved_chunks, start=1):
        print(f"Source {i}")
        print(item["chunk"][:500])
        print("-" * 80)

In [33]:
embeddings = embedding_model.encode(chunks, normalize_embeddings=True)

embeddings_np = np.array(embeddings).astype("float32")

dimension = embeddings_np.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings_np)

In [34]:
def retrieve_chunks(question, top_k=5):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    question_embedding_np = np.array(question_embedding).astype("float32")

    scores, indices = index.search(question_embedding_np, k=top_k)

    retrieved = []

    for i, idx in enumerate(indices[0]):
        retrieved.append({
            "chunk": chunks[idx],
            "score": scores[0][i]
        })

    return retrieved

In [35]:
question = "What is the objective of the project?"

retrieved_chunks = retrieve_chunks(question, top_k=5)

for i, item in enumerate(retrieved_chunks, start=1):
    print(f"Source {i}")
    print("Score:", item["score"])
    print(item["chunk"][:700])
    print("-" * 80)

Source 1
Score: 0.14853115
Eastern Mediterranean University School of Computing and Technology Claud Computing Applications Prepared by Abdulrahman Ahmed 22902545 Osman Eid 23704362 Mohamed Allafi 22901510 Abdulmalik Khalid 22901037 Ali Mohammed 23704831 Section One: Project Objective The objective of this project is to design, build, and test a simple RESTful Web Service using ASP.NET Core Web API. The application chosen for this assignment is a Clinic Appointment Management API, named ClinicApi, which allows a client to manage patient appointments through a set of HTTP endpoints. The main goal of the project is to understand how a Web API is structured internally — from the data model, to the business logic layer, t
--------------------------------------------------------------------------------
Source 2
Score: 0.1430089
● Visual Studio / Visual Studio Code – development environment used to write and debug the project. ● .NET SDK – used to compile and run the API on a local Kestrel w

In [36]:
def answer_question(question, top_k=3):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)

    context = retrieved_chunks[0]["chunk"]

    print("Question:", question)
    print("\nAnswer based on document:\n")
    print(context[:700])

    print("\nSources:")
    for i, item in enumerate(retrieved_chunks, start=1):
        print(f"Source {i} | Score: {item['score']:.4f}")

In [37]:
def generate_simple_answer(question, top_k=3):
    retrieved_chunks = retrieve_chunks(question, top_k=top_k)

    context = "\n\n".join([item["chunk"] for item in retrieved_chunks])

    print("Question:", question)
    print("\nRetrieved Context:")
    print(context[:1500])

    print("\nSuggested Answer:")
    print("Based on the document, write a short answer using the retrieved context above.")

In [38]:
generate_simple_answer("What is the objective of the project?")

Question: What is the objective of the project?

Retrieved Context:
Eastern Mediterranean University School of Computing and Technology Claud Computing Applications Prepared by Abdulrahman Ahmed 22902545 Osman Eid 23704362 Mohamed Allafi 22901510 Abdulmalik Khalid 22901037 Ali Mohammed 23704831 Section One: Project Objective The objective of this project is to design, build, and test a simple RESTful Web Service using ASP.NET Core Web API. The application chosen for this assignment is a Clinic Appointment Management API, named ClinicApi, which allows a client to manage patient appointments through a set of HTTP endpoints. The main goal of the project is to understand how a Web API is structured internally — from the data model, to the business logic layer, to the controller that exposes the endpoints — and to learn how to test and verify that a REST API behaves correctly using an API testing tool such as Postman. The project demonstrates the full lifecycle of a resource (an appointment